# Capital Bikeshare Network Optimization & Crew Deployment

**Python · pandas · K-Means · K-Medians · K-Medoids · Hierarchical Clustering**

## Business objective

This project uses 2025 Capital Bikeshare station locations to explore:

> **How should stations be grouped into geographic service territories, and which existing stations could serve as candidate crew depots?**

The source assignment loaded **12 monthly files with 6,662,647 trip records**. This portfolio version removes quiz questions, point values, and Google Drive-specific code while preserving the applied clustering analysis and source-reported outputs.


## Executive summary

The source analysis:

- evaluated K-Means for **k=2 to 50**;
- selected **k=10** from the elbow/WSS curve;
- compared K-Means, K-Medians, K-Medoids, and hierarchical clustering;
- used K-Medoids to identify **10 actual stations** as candidate depots.

K-Medoids is operationally attractive because each medoid is a real station rather than an artificial centroid.

This remains a proof of concept. Final deployment would require street-network travel time, demand/workload, station capacity, staffing, and cost constraints.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

K_STAR = 10


## 1. Load 2025 Capital Bikeshare data

Place extracted 2025 trip-history CSV files in `data/raw/`. The source run reported **12 files, 6,662,647 rows, and 13 columns**.


In [ ]:
trip_files = sorted(DATA_DIR.glob("*capitalbikeshare-tripdata*.csv"))

if not trip_files:
    raise FileNotFoundError(
        "No Capital Bikeshare CSVs found in data/raw/. See data/README.md."
    )

df = pd.concat(
    (pd.read_csv(path, low_memory=False) for path in trip_files),
    ignore_index=True,
)

print(f"Loaded {len(trip_files)} files")
print(f"Shape: {df.shape}")
df.head()


## 2. Create station-location table

The source notebook extracts unique **start station** names and retains one latitude/longitude pair for each station.


In [ ]:
stations = (
    df[["start_station_name","start_lat","start_lng"]]
    .dropna()
    .drop_duplicates()
    .groupby("start_station_name", as_index=False)
    .first()
    .rename(columns={"start_lat":"Latitude","start_lng":"Longitude"})
)

X = stations[["Latitude","Longitude"]]
print("Unique stations:", len(stations))
stations.head()


In [ ]:
plt.figure(figsize=(10,8))
plt.scatter(stations["Longitude"], stations["Latitude"], alpha=0.6, s=18)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Capital Bikeshare Station Locations")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


![Source-executed station map](../images/station_locations.png)


## 3. Select crew count with the K-Means elbow curve

The source notebook fits K-Means from **k=2 to 50** and selects **k*=10** because the WSS improvement begins to level off around that point.


In [ ]:
wss = []
k_values = range(2,51)

for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X)
    wss.append(model.inertia_)

plt.figure(figsize=(12,6))
plt.plot(list(k_values), wss, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-cluster sum of squares (WSS)")
plt.title("Elbow Method")
plt.xticks(np.arange(2,51,2))
plt.grid(True)
plt.tight_layout()
plt.show()


![Source-executed elbow curve](../images/elbow_wss.png)

**Source choice: k = 10 crews.** This is a practical elbow judgment rather than a unique mathematical optimum.


## 4. K-Means service territories

In [ ]:
kmeans = KMeans(n_clusters=K_STAR, n_init=10, random_state=42)
stations["Cluster_KMeans"] = kmeans.fit_predict(X)

plt.figure(figsize=(10,8))
scatter = plt.scatter(
    stations["Longitude"], stations["Latitude"],
    c=stations["Cluster_KMeans"], cmap="tab10", alpha=0.6, s=20
)
plt.legend(*scatter.legend_elements(), title="Clusters")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"K-Means Clusters (k={K_STAR})")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


![Source-executed K-Means clusters](../images/kmeans_clusters.png)

The source notebook describes these territories as relatively compact and easy to interpret.


## 5. K-Medians service territories

The source assignment motivates K-Medians as an L1/Manhattan-distance alternative to squared-Euclidean K-Means.


In [ ]:
from pyclustering.cluster.kmedians import kmedians
from pyclustering.cluster.center_initializer import kmeans_plusplus_initializer

if not hasattr(np, "warnings"):
    np.warnings = warnings

data = X.values.tolist()
initial_centers = kmeans_plusplus_initializer(data, K_STAR).initialize()

kmedians_model = kmedians(data, initial_centers)
kmedians_model.process()
clusters = kmedians_model.get_clusters()

labels = np.zeros(len(stations), dtype=int)
for cluster_id, indices in enumerate(clusters):
    labels[indices] = cluster_id

stations["Cluster_KMedians"] = labels

plt.figure(figsize=(10,8))
scatter = plt.scatter(
    stations["Longitude"], stations["Latitude"],
    c=stations["Cluster_KMedians"], cmap="tab10", alpha=0.6, s=20
)
plt.legend(*scatter.legend_elements(), title="Clusters")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"K-Medians Clusters (k={K_STAR})")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


![Source-executed K-Medians clusters](../images/kmedians_clusters.png)


## 6. K-Medoids and candidate depots

K-Medoids is especially practical here because a medoid is an **actual station**. The source notebook therefore treats the selected medoids as candidate crew depots.


In [ ]:
from pyclustering.cluster.kmedoids import kmedoids

initial_medoids = kmeans_plusplus_initializer(
    data, K_STAR
).initialize(return_index=True)

kmedoids_model = kmedoids(data, initial_medoids)
kmedoids_model.process()

clusters_medoids = kmedoids_model.get_clusters()
medoid_indices = kmedoids_model.get_medoids()

labels_medoids = np.zeros(len(stations), dtype=int)
for cluster_id, indices in enumerate(clusters_medoids):
    labels_medoids[indices] = cluster_id

stations["Cluster_KMedoids"] = labels_medoids
candidate_depots = stations.iloc[medoid_indices].copy()

plt.figure(figsize=(10,8))
plt.scatter(
    stations["Longitude"], stations["Latitude"],
    c=stations["Cluster_KMedoids"], cmap="tab10", alpha=0.4, s=20
)
plt.scatter(
    candidate_depots["Longitude"], candidate_depots["Latitude"],
    marker="*", s=180, label="Candidate depots"
)
plt.legend()
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"K-Medoids Clusters and Candidate Depots (k={K_STAR})")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

candidate_depots[["start_station_name","Latitude","Longitude"]]


### Source-reported candidate depots

| Station | Latitude | Longitude |
|---|---:|---:|
| Rockville Metro West | 39.084379 | -77.146866 |
| Commonwealth Ave & E Monroe Ave | 38.820058 | -77.062821 |
| Reston Town Center Metro North | 38.953691 | -77.359717 |
| Stadium Armory Metro | 38.885483 | -76.977187 |
| Pimmit Dr & Los Pueblos Ln | 38.900371 | -77.205428 |
| Washington Blvd & 7th St N | 38.880810 | -77.090792 |
| Western Ave & Pinehurst Cir NW | 38.975739 | -77.066409 |
| Riggs Rd & East West Hwy | 38.972500 | -76.980700 |
| 14th St & Rhode Island Ave NW | 38.908600 | -77.032300 |
| Fair Woods Pkwy & Fairfax Blvd | 38.862804 | -77.293922 |

![Source-executed K-Medoids depots](../images/kmedoids_depots.png)

These are **candidate depots from the source clustering run**, not deployment-validated facilities.


## 7. Hierarchical clustering

In [ ]:
hierarchical = AgglomerativeClustering(
    n_clusters=K_STAR, linkage="ward"
)
stations["Cluster_Hierarchical"] = hierarchical.fit_predict(X)

plt.figure(figsize=(10,8))
scatter = plt.scatter(
    stations["Longitude"], stations["Latitude"],
    c=stations["Cluster_Hierarchical"], cmap="tab10", alpha=0.6, s=20
)
plt.legend(*scatter.legend_elements(), title="Clusters")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Hierarchical Clusters (k={K_STAR})")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


![Source-executed hierarchical clusters](../images/hierarchical_clusters.png)


## 8. Method comparison

| Method | Source-notebook interpretation | Operational implication |
|---|---|---|
| K-Means | Compact, approximately spherical groups | Simple territories; centroid may not be a station |
| K-Medians | L1 / Manhattan-style objective | More robust to extreme locations |
| K-Medoids | Center is an observed station | Strongest source-notebook option for candidate physical depots |
| Hierarchical | Successive agglomerative merges | Alternative view of geographic grouping |

The source assignment favors **K-Medoids** for practical crew deployment because it selects an existing station as each center.


## 9. Practical deployment criteria from the source notebook

The source analysis recommends evaluating:

1. **Operational feasibility** of each proposed base
2. **Travel efficiency** between depots and assigned stations
3. **Workload balance** across crews
4. **Pilot / shadow testing** before full deployment


## 10. Portfolio limitations and next steps

These are professional extensions to the source assignment:

- Distance on raw latitude/longitude is only an approximation of real street-network travel.
- `k=10` is based on a visual elbow and should also be tested against staffing cost and service targets.
- All stations are currently weighted equally instead of by demand, capacity, or maintenance burden.
- Cluster sizes do not guarantee balanced crew workload.
- Candidate medoid stations may not have the physical capacity to serve as crew depots.
- A production system should add road-network travel time and an optimization layer for routing/rebalancing.

A strong next version would combine these territories with demand forecasts, travel-time matrices, crew-capacity constraints, and route optimization.


## 11. AI-assistance disclosure

Generative AI tools were used during the original coursework to assist with code drafting/debugging and written explanations. Source outputs and interpretations were reviewed by the author. This portfolio version removes classroom-only content and reorganizes the applied analysis for professional presentation.
